# Module 1: Foundations & Tools

**Goal:** Master the basics of LLM Engineering: API interactions, Prompt Engineering, Structured Outputs, and Tool Use (Function Calling).

## 1. Setup
We use `python-dotenv` to manage API keys. Ensure you have a `.env` file in the project root.

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI

# Load environment variables
load_dotenv(override=True)

api_key = os.getenv('OPENAI_API_KEY')
if api_key:
    print(f"API Key found: {api_key[:8]}...")
else:
    print("Warning: OPENAI_API_KEY not found in environment.")

# Initialize Client
client = OpenAI(api_key=api_key)

API Key found: sk-proj-...


## 2. Basic Chat Completion
The core interaction loop.

In [2]:
MODEL = "gpt-4o-mini"  # Use a cost-effective model

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful AI assistant experienced in Python engineering."},
        {"role": "user", "content": "Explain the difference between concurrent.futures and asyncio in one sentence."}
    ]
)

print(response.choices[0].message.content)

`concurrent.futures` is designed for parallel execution of threads or processes, while `asyncio` provides an asynchronous I/O framework that facilitates writing single-threaded concurrent code using coroutines and event loops.


## 3. Tool Use (Funciton Calling)
LLMs can't natively calculate math or fetch data. We provide "tools" (functions) they can call.
Here we define a simple dummy tool to calculate ticket prices.

In [3]:
import json

def get_ticket_price(destination: str, class_type: str = "economy") -> str:
    """Dummy function to get ticket price."""
    print(f"\n[Tool Called] get_ticket_price(destination={destination}, class_type={class_type})")
    
    prices = {
        "london": {"economy": 500, "business": 1500},
        "paris": {"economy": 450, "business": 1200},
        "tokyo": {"economy": 800, "business": 2500}
    }
    
    city = destination.lower()
    if city in prices:
        price = prices[city].get(class_type.lower(), 500)
        return json.dumps({"price": price, "currency": "USD"})
    return json.dumps({"error": "City not found"})

# Tool Definition Schema
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_ticket_price",
            "description": "Get the price of a flight ticket to a specific destination.",
            "parameters": {
                "type": "object",
                "properties": {
                    "destination": {"type": "string", "description": "The city name (e.g. London, Paris)"},
                    "class_type": {"type": "string", "enum": ["economy", "business"], "default": "economy"}
                },
                "required": ["destination"]
            }
        }
    }
]

## 4. The Agent Loop
We need a loop to:
1. Send user query + tools to LLM.
2. Check if LLM wants to call a tool.
3. Execute tool if requested.
4. Send tool result back to LLM.
5. Get final response.

In [4]:
def run_agent(user_query: str):
    messages = [
        {"role": "system", "content": "You are a flight booking assistant. Use tools to check prices."},
        {"role": "user", "content": user_query}
    ]

    # First Turn
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools,
        tool_choice="auto"
    )
    
    msg = response.choices[0].message
    messages.append(msg)

    # Check for tool calls
    if msg.tool_calls:
        for tool_call in msg.tool_calls:
            func_name = tool_call.function.name
            args = json.loads(tool_call.function.arguments)
            
            if func_name == "get_ticket_price":
                result = get_ticket_price(**args)
                
                # Append result to messages
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": result
                })
        
        # Second Turn (Final Answer)
        final_response = client.chat.completions.create(
            model=MODEL,
            messages=messages
        )
        return final_response.choices[0].message.content
        
    return msg.content

# Test it
print(run_agent("How much is a business class ticket to Tokyo?"))


[Tool Called] get_ticket_price(destination=Tokyo, class_type=business)
A business class ticket to Tokyo is approximately $2,500 USD.


## 5. Gradio UI
Let's wrap this in a simple web UI using `gradio`.

In [5]:
import gradio as gr

def chat_interface(message, history):
    return run_agent(message)

if __name__ == "__main__":
    # Import our custom styles from reference code
    try:
        # Assuming this notebook is in revision_canonical/
        from reference_code.styles import CSS
        print("Loaded custom styles from reference_code")
    except ImportError as e:
        print(f"Could not load styles: {e}")
        CSS = ""

    demo = gr.ChatInterface(
        fn=chat_interface,
        title="FlightAI Assistant",
        description="Ask about flight prices to London, Paris, or Tokyo.",
        css=CSS
    )
    
    # Launching with share=False for local dev
    demo.launch(share=False)

Loaded custom styles from reference_code


/Users/electrovese/Documents/chandan/R&D/ai/LLM/llm_engineering/.venv/lib/python3.12/site-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.



[Tool Called] get_ticket_price(destination=London, class_type=economy)
